# Voice Single Benchmark: OmniVoice vs ElevenLabs

**Mục đích:** So sánh chất lượng giọng đọc giữa OmniVoice (Colab T4) và ElevenLabs với audio dài >1 phút.

**Quy trình:**
1. Cài đặt dependencies + kiểm tra GPU + Clone repo
2. Load voice data (audio ElevenLabs + text nội dung)
3. Sinh audio bằng OmniVoice với cùng text
4. So sánh waveform (sóng âm)
5. So sánh spectrogram + pitch contour
6. Tính quality metrics (spectral, MFCC, energy)
7. Tổng hợp + kết luận

**Repo:** github.com/doanquangkien/voice-notebooks (PUBLIC)

**Chạy trên Colab:** Chỉ cần chọn GPU T4 và chạy tất cả cells. Không cần upload files.

---

## Cell 1: Cài đặt + GPU Check

In [ ]:
# Cell 1: Install dependencies + GPU check + Download voice data
print('Đang cài đặt (~2 phút)...')
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"
!pip install -q librosa matplotlib soundfile ipython
print('Cài đặt hoàn tất!')

# GPU check
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    raise RuntimeError('KHÔNG TÌM THẤY GPU! Bật GPU trong Runtime > Change runtime type')

# Download voice data trực tiếp từ GitHub (không cần clone repo)
import os, urllib.request

os.makedirs('voice_data', exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/doanquangkien/voice-notebooks/main/analysis/single_voice_benchmark/voice_data'

files_to_download = [
    ('voice_data/elevenlabs_audio.wav', f'{BASE_URL}/elevenlabs_audio.wav'),
    ('voice_data/text.txt', f'{BASE_URL}/text.txt'),
]

for local_path, url in files_to_download:
    if not os.path.exists(local_path):
        print(f'Downloading {local_path}...')
        urllib.request.urlretrieve(url, local_path)
    else:
        print(f'Already exists: {local_path}')

# Verify
print(f'\nVoice data:')
for f in os.listdir('voice_data'):
    size = os.path.getsize(f'voice_data/{f}') / 1024
    print(f'  {f}: {size:.1f} KB')

## Cell 2: Load Voice Data

**Files được tải tự động trong Cell 1:**
- `voice_data/elevenlabs_audio.wav` — Audio từ ElevenLabs (48kHz, 83.5s)
- `voice_data/text.txt` — Text nội dung đọc

Không cần upload bất cứ thứ gì.

In [ ]:
# Cell 2: Load voice data
import os

# === CẤU HÌNH ===
VOICE_KEY = 'srt_voice'  # Tên giọng: SRT feature promo
ELEVENLABS_AUDIO = 'voice_data/elevenlabs_audio.wav'  # Audio từ ElevenLabs (48kHz)
VOICE_TEXT_FILE = 'voice_data/text.txt'  # Text nội dung

# Kiểm tra files tồn tại
if not os.path.exists(ELEVENLABS_AUDIO):
    raise FileNotFoundError(f'Không tìm thấy audio: {ELEVENLABS_AUDIO}')
if not os.path.exists(VOICE_TEXT_FILE):
    raise FileNotFoundError(f'Không tìm thấy text: {VOICE_TEXT_FILE}')

# Đọc text
with open(VOICE_TEXT_FILE, 'r', encoding='utf-8') as f:
    voice_text = f.read().strip()

# Load audio info
import librosa
y_el, sr_el = librosa.load(ELEVENLABS_AUDIO, sr=None)
duration_el = len(y_el) / sr_el

print(f'=== VOICE DATA ===')
print(f'Giọng: {VOICE_KEY}')
print(f'Audio ElevenLabs: {ELEVENLABS_AUDIO}')
print(f'Thời lượng: {duration_el:.1f} giây ({duration_el/60:.1f} phút)')
print(f'Sample rate: {sr_el} Hz')
print(f'Text length: {len(voice_text)} ký tự')
print(f'\nText preview:')
print(voice_text[:200] + '...' if len(voice_text) > 200 else voice_text)

## Cell 3: Generate OmniVoice Audio

In [ ]:
# Cell 3: Generate OmniVoice audio
import torch as _torch
if not hasattr(_torch, '_utils'):
    _torch._utils = _torch._C._utils

import transformers as _tf
class _SafeAutoFeatureExtractor:
    @staticmethod
    def from_pretrained(model_name, **kwargs):
        try:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True, **kwargs)
            sr = getattr(cfg, 'sampling_rate', 24000)
        except Exception:
            sr = 24000
        class _Result:
            sampling_rate = sr
        return _Result()
_tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.utils.common import get_best_device
import soundfile as sf
import numpy as np

print('Loading OmniVoice model...')
DEVICE = get_best_device()
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
print(f'Model loaded! SR: {SAMPLING_RATE}Hz')

# Config — có thể thử các config khác nhau
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=32, guidance_scale=1.8,
    denoise=True, preprocess_prompt=True, postprocess_output=True,
    position_temperature=5.0, class_temperature=0.2,
    pad_duration=0.1, fade_duration=0.1,
)
CONFIG = {'steps': 32, 'guidance_scale': 1.8, 'speed': 0.95}

# Tạo voice prompt từ audio ElevenLabs
print(f'\nCreating voice prompt from: {ELEVENLABS_AUDIO}')
voice_prompt = model.create_voice_clone_prompt(ref_audio=ELEVENLABS_AUDIO)

# Generate
print(f'Generating OmniVoice audio...')
print(f'Config: steps={CONFIG["steps"]}, guidance_scale={CONFIG["guidance_scale"]}, speed={CONFIG["speed"]}')

audio = model.generate(
    text=voice_text, voice_clone_prompt=voice_prompt,
    language='vi', speed=CONFIG['speed'], generation_config=GEN_CFG
)[0]

# Save
os.makedirs('output', exist_ok=True)
output_path = f'output/{VOICE_KEY}_omnivoice.wav'
sf.write(output_path, audio, SAMPLING_RATE)

duration_ov = len(audio) / SAMPLING_RATE
print(f'\nSaved: {output_path}')
print(f'Thời lượng: {duration_ov:.1f} giây ({duration_ov/60:.1f} phút)')
print(f'\nSo sánh:')
print(f'  ElevenLabs: {duration_el:.1f}s')
print(f'  OmniVoice:  {duration_ov:.1f}s')
print(f'  Delta:      {duration_ov - duration_el:+.1f}s ({(duration_ov/duration_el - 1)*100:+.1f}%)')

## Cell 4: Waveform Comparison

In [ ]:
# Cell 4: Waveform comparison
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

def load_audio(path, sr=24000):
    y, orig_sr = librosa.load(path, sr=sr)
    return y, sr

y_el, sr_el = load_audio(ELEVENLABS_AUDIO)
y_ov, sr_ov = load_audio(f'output/{VOICE_KEY}_omnivoice.wav')

fig, axes = plt.subplots(2, 1, figsize=(16, 8))
fig.suptitle(f'Waveform Comparison: {VOICE_KEY}\nElevenLabs vs OmniVoice', fontsize=14, fontweight='bold')

# ElevenLabs
librosa.display.waveshow(y_el, sr=sr_el, ax=axes[0], color='#2196F3')
axes[0].set_title(f'ElevenLabs ({len(y_el)/sr_el:.1f}s)', fontsize=12)
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim(0, max(len(y_el)/sr_el, len(y_ov)/sr_ov))

# OmniVoice
librosa.display.waveshow(y_ov, sr=sr_ov, ax=axes[1], color='#FF9800')
axes[1].set_title(f'OmniVoice ({len(y_ov)/sr_ov:.1f}s)', fontsize=12)
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, max(len(y_el)/sr_el, len(y_ov)/sr_ov))

plt.tight_layout()
plt.savefig('output/comparison_waveform.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/comparison_waveform.png')

## Cell 5: Spectrogram + Pitch Contour

In [ ]:
# Cell 5: Spectrogram + pitch contour
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle(f'Spectrogram + Pitch: {VOICE_KEY}\nElevenLabs vs OmniVoice', fontsize=14, fontweight='bold')

# Mel Spectrogram — ElevenLabs
S_el = librosa.feature.melspectrogram(y=y_el, sr=sr_el, n_mels=80)
S_el_db = librosa.power_to_db(S_el, ref=np.max)
librosa.display.specshow(S_el_db, sr=sr_el, x_axis='time', y_axis='mel', ax=axes[0, 0], cmap='magma')
axes[0, 0].set_title('Mel Spectrogram — ElevenLabs', fontsize=11)

# Mel Spectrogram — OmniVoice
S_ov = librosa.feature.melspectrogram(y=y_ov, sr=sr_ov, n_mels=80)
S_ov_db = librosa.power_to_db(S_ov, ref=np.max)
librosa.display.specshow(S_ov_db, sr=sr_ov, x_axis='time', y_axis='mel', ax=axes[0, 1], cmap='magma')
axes[0, 1].set_title('Mel Spectrogram — OmniVoice', fontsize=11)

# Pitch Contour — ElevenLabs
f0_el = librosa.yin(y_el, fmin=60, fmax=400, sr=sr_el)
times_el = librosa.times_like(f0_el, sr=sr_el)
axes[1, 0].plot(times_el, f0_el, color='#2196F3', linewidth=0.8)
axes[1, 0].set_title('Pitch Contour — ElevenLabs', fontsize=11)
axes[1, 0].set_ylabel('Hz')
axes[1, 0].set_ylim(60, 400)
axes[1, 0].grid(True, alpha=0.3)

# Pitch Contour — OmniVoice
f0_ov = librosa.yin(y_ov, fmin=60, fmax=400, sr=sr_ov)
times_ov = librosa.times_like(f0_ov, sr=sr_ov)
axes[1, 1].plot(times_ov, f0_ov, color='#FF9800', linewidth=0.8)
axes[1, 1].set_title('Pitch Contour — OmniVoice', fontsize=11)
axes[1, 1].set_ylabel('Hz')
axes[1, 1].set_ylim(60, 400)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/comparison_spectrogram_pitch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: output/comparison_spectrogram_pitch.png')

## Cell 6: Quality Metrics

In [ ]:
# Cell 6: Quality metrics
import pandas as pd

def compute_metrics(y, sr):
    """Tính các audio features"""
    # RMS Energy
    rms = np.sqrt(np.mean(y**2))
    
    # Spectral Centroid (độ sáng)
    sc = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    
    # Spectral Bandwidth (độ rộng băng tần)
    sb = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
    
    # Zero Crossing Rate
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    
    # MFCC (đặc trưng âm sắc)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    
    # Pitch statistics
    f0 = librosa.yin(y, fmin=60, fmax=400, sr=sr)
    f0_voiced = f0[f0 > 60]  # Chỉ lấy pitch có âm
    pitch_mean = np.mean(f0_voiced) if len(f0_voiced) > 0 else 0
    pitch_std = np.std(f0_voiced) if len(f0_voiced) > 0 else 0
    
    # Duration
    duration = len(y) / sr
    
    return {
        'duration_s': round(duration, 2),
        'rms_energy': round(rms, 4),
        'spectral_centroid_hz': round(sc, 1),
        'spectral_bandwidth_hz': round(sb, 1),
        'zero_crossing_rate': round(zcr, 4),
        'pitch_mean_hz': round(pitch_mean, 1),
        'pitch_std_hz': round(pitch_std, 1),
        'mfcc_0': round(mfcc_mean[0], 2),
        'mfcc_1': round(mfcc_mean[1], 2),
        'mfcc_2': round(mfcc_mean[2], 2),
    }

# Tính metrics
m_el = compute_metrics(y_el, sr_el)
m_ov = compute_metrics(y_ov, sr_ov)

# Tạo bảng so sánh
rows = [
    {'source': 'ElevenLabs', **m_el},
    {'source': 'OmniVoice', **m_ov},
]
df = pd.DataFrame(rows)

# Hiển thị
print('=' * 80)
print(f'VOICE QUALITY METRICS — {VOICE_KEY}')
print('=' * 80)
print(df.to_string(index=False))

# Tính delta
print('\n' + '=' * 80)
print('DELTA (OmniVoice - ElevenLabs)')
print('Am = ít hơn, Dương = nhiều hơn')
print('=' * 80)

delta = {'voice': VOICE_KEY}
for col in ['duration_s', 'rms_energy', 'spectral_centroid_hz', 'spectral_bandwidth_hz',
            'zero_crossing_rate', 'pitch_mean_hz', 'pitch_std_hz']:
    d = m_ov[col] - m_el[col]
    delta[col] = round(d, 3)
    sign = '+' if d > 0 else ''
    unit = 's' if 'duration' in col else 'Hz' if 'hz' in col.lower() or 'pitch' in col else ''
    print(f'  {col:30s}: {sign}{d:.3f} {unit}')

# Save
df.to_csv('output/metrics_comparison.csv', index=False)
pd.DataFrame([delta]).to_csv('output/metrics_delta.csv', index=False)
print('\nSaved: output/metrics_comparison.csv, output/metrics_delta.csv')

## Cell 7: Tổng hợp + Kết luận

In [ ]:
# Cell 7: Tổng hợp
print('=' * 80)
print(f'TỔNG HỢP KẾT QUẢ BENCHMARK — {VOICE_KEY}')
print('=' * 80)

print(f'\nConfig: steps={CONFIG["steps"]}, guidance_scale={CONFIG["guidance_scale"]}, speed={CONFIG["speed"]}')

print(f'\n--- PHÂN TICH ---')

# Duration
dur_delta = m_ov['duration_s'] - m_el['duration_s']
dur_pct = (m_ov['duration_s'] / m_el['duration_s'] - 1) * 100
print(f'\n1. Duration: {m_el["duration_s"]:.1f}s (EL) vs {m_ov["duration_s"]:.1f}s (OV) = {dur_delta:+.1f}s ({dur_pct:+.1f}%)')
if abs(dur_pct) > 10:
    print(f'   → Chênh lệch lớn. Điều chỉnh speed.')
else:
    print(f'   → Chênh lệch chấp nhận được.')

# Pitch
pitch_delta = m_ov['pitch_mean_hz'] - m_el['pitch_mean_hz']
print(f'\n2. Pitch Mean: {m_el["pitch_mean_hz"]:.1f}Hz (EL) vs {m_ov["pitch_mean_hz"]:.1f}Hz (OV) = {pitch_delta:+.1f}Hz')
if abs(pitch_delta) > 20:
    print(f'   → Chênh lệch lớn. Thử điều chỉnh guidance_scale.')
else:
    print(f'   → Pitch tương đồng.')

# Pitch Std (emotion)
std_delta = m_ov['pitch_std_hz'] - m_el['pitch_std_hz']
print(f'\n3. Pitch Std (biểu cảm): {m_el["pitch_std_hz"]:.1f}Hz (EL) vs {m_ov["pitch_std_hz"]:.1f}Hz (OV) = {std_delta:+.1f}Hz')
if std_delta < -10:
    print(f'   → OmniVoice "phẳng" hơn. Thử tăng guidance_scale hoặc steps.')
elif std_delta > 10:
    print(f'   → OmniVoice "nhảy" hơn. Thử giảm guidance_scale.')
else:
    print(f'   → Mức biểu cảm tương đương.')

# Spectral Centroid
sc_delta = m_ov['spectral_centroid_hz'] - m_el['spectral_centroid_hz']
print(f'\n4. Spectral Centroid (độ sáng): {m_el["spectral_centroid_hz"]:.1f}Hz (EL) vs {m_ov["spectral_centroid_hz"]:.1f}Hz (OV) = {sc_delta:+.1f}Hz')
if sc_delta > 100:
    print(f'   → OmniVoice sáng/cao hơn.')
elif sc_delta < -100:
    print(f'   → OmniVoice trầm hơn.')
else:
    print(f'   → Tương đương.')

print(f'\n--- ĐỀ XUẤT CONFIG TIẾP THEO ---')
print(f'Thử các config sau để so sánh:')
print(f'')
print(f'CONFIG_A = {{"steps": 48, "guidance_scale": 1.8, "speed": 0.95}}  # Nhiều steps hơn')
print(f'CONFIG_B = {{"steps": 32, "guidance_scale": 2.5, "speed": 0.95}}  # Cao guidance')
print(f'CONFIG_C = {{"steps": 32, "guidance_scale": 1.8, "speed": 0.85}}  # Chậm hơn')
print(f'CONFIG_D = {{"steps": 48, "guidance_scale": 2.5, "speed": 0.90}}  # Combo')
print(f'')
print(f'Sau khi chạy 4 configs → đối chiếu metrics → chọn config tốt nhất.')

print(f'\n' + '=' * 80)
print(f'KẾT THÚC BENCHMARK')
print(f'Files output:')
print(f'  - output/{VOICE_KEY}_omnivoice.wav  (audio OmniVoice)')
print(f'  - output/comparison_waveform.png    (biểu đồ sóng âm)')
print(f'  - output/comparison_spectrogram_pitch.png (biểu đồ tần số + pitch)')
print(f'  - output/metrics_comparison.csv     (bảng so sánh)')
print(f'  - output/metrics_delta.csv          (bảng delta)')
print(f'=' * 80)

## Cell 8: Zip + Download

In [ ]:
# Cell 8: Zip + download
import zipfile
from google.colab import files as colab_files

zip_name = f'{VOICE_KEY}_benchmark_results.zip'

# Files cần zip
output_files = []

# Audio
wav_path = f'output/{VOICE_KEY}_omnivoice.wav'
if os.path.exists(wav_path):
    output_files.append(wav_path)

# Charts
for f in ['output/comparison_waveform.png', 'output/comparison_spectrogram_pitch.png']:
    if os.path.exists(f):
        output_files.append(f)

# CSV
for f in ['output/metrics_comparison.csv', 'output/metrics_delta.csv']:
    if os.path.exists(f):
        output_files.append(f)

# Zip
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in output_files:
        zf.write(fpath)
        print(f'  + {fpath}')

zip_size = os.path.getsize(zip_name) / 1024 / 1024
print(f'\n{zip_name}: {zip_size:.1f} MB ({len(output_files)} files)')

# Download
print('\nĐang tải xuống...')
colab_files.download(zip_name)
print('Hoàn tất!')